# पाठ 11 - एजंट ते एजंट (A2A) प्रोटोकॉल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A प्रोटोकॉल म्हणजे काय?

हे **एजंट-टू-एजंट (A2A) प्रोटोकॉल** एक मुक्त मानक आहे जे AI एजंटांना संवाद साधण्यास,
एकमेकांना शोधण्यास, आणि सहकार्य करण्यास — अगदी ते वेगवेगळ्या फ्रेमवर्कवर तयार केलेले किंवा होस्ट केलेले
वेगवेगळ्या सेवांवर असले तरीही.

मुख्य संकल्पना:

- **शोध** – एजंट आपल्या क्षमता वर्णन करणारा *Agent Card* प्रकाशित करतात, ज्यामुळे
  इतर एजंट्स (किंवा ऑर्केस्ट्रेटर्स) एका कार्यासाठी योग्य विशेषज्ञ शोधणे सोपे होते.
- **संदेश देवाणघेवाण** – एजंट एकसारख्या प्रोटोकॉलद्वारे संरचित संदेशांची देवाणघेवाण करतात, त्यामुळे एक
  एका एजंटकडून येणारी विनंती दुसऱ्या एजंटद्वारे समजली जाऊ शकते आणि पूर्ण केली जाऊ शकते, त्याच्या अंतर्गत
  अमलबजावणीवर अवलंबून न राहता.
- **टास्क जीवनचक्र** – A2A असे राज्य परिभाषित करते जसे की *submitted*, *working*, *completed*, आणि
  *failed*, ज्यामुळे ऑर्केस्ट्रेटरला दिलेल्या कामाची प्रगती कशी होत आहे याचे संपूर्ण दर्शन मिळते.

या धड्यात आम्ही A2A-शैलीचे सहकार्य अनुकरण करतो तीन विशेषीकृत प्रवास एजंट
एक कार्यप्रवाहात जिथे प्रत्येक एजंट आपले कौशल्य देतो आणि निकाल पुढील एजंटकडे सोपवतो.


## विशेषीकृत प्रवास प्रतिनिधी तयार करणे


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## मल्टी-एजंट सहकार्य वर्कफ्लोद्वारे

आम्ही तीन एजंटना अनुक्रमिक वर्कफ्लोमध्ये जोडतो जे A2A संदेश विनिमयाचे प्रतिबिंब आहे:

1. **CurrencyExchangeAgent** वापरकर्त्याची विनंती प्राप्त करतो आणि चलन मार्गदर्शन तयार करतो.
2. **ActivityPlannerAgent** समृद्ध संदर्भ प्राप्त करतो आणि क्रियाकलाप शिफारसी जोडतो.
3. **TravelManagerAgent** दोन्ही इनपुट एकत्र करून अंतिम प्रवास सारांश तयार करतो.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## उत्पादनात A2A समजून घेणे

उत्पादन वातावरणात A2A प्रोटोकॉल सामर्थ्यवान क्रॉस-सेवा परिदृश्य उघडतो:

| क्षमता | वर्णन |
|---|---|
| **फ्रेमवर्क-आधारित अंतःसहकार्य** | एखाद्या फ्रेमवर्कने तयार केलेला एजंट कोणत्याही अन्य A2A-अनुपालन करणाऱ्या फ्रेमवर्कने तयार केलेल्या एजंटला कार्ये सोपवू शकतो, ज्यामुळे खऱ्या अर्थाने क्रॉस-संगठन परस्पर-संवाद सक्षम होतो. |
| **सेवा सीमा** | एजंट स्वतंत्र मायक्रोसर्व्हिसेस, क्लाउड प्रदेश किंवा अगदी वेगवेगळ्या संस्थांमध्ये असू शकतात तरीही ते सुरळीतपणे सहकार्य करू शकतात. |
| **गतिशील शोध** | एक ऑर्केस्ट्रेटर रनटाइममध्ये Agent Card रजिस्ट्रीवर क्वेरी करून एखाद्या उपकार्यासाठी सर्वात योग्य विशेषज्ञ शोधू शकतो. |
| **स्ट्रीमिंग & पुश सूचना** | A2A रिअल-टाइम प्रगती अद्यतनांसाठी Server-Sent Events (SSE) आणि दीर्घकालीन कार्यांसाठी पुश सूचनांना समर्थन देते. |

वरील आपण तयार केलेला वर्कफ्लो या पॅटर्नचा साधी, इन-प्रोसेस आवृत्ती आहे. प्रत्यक्ष तैनातीमध्ये प्रत्येक एजंट एक HTTP endpoint प्रदर्शित करेल, Agent Card प्रकाशित करेल आणि A2A JSON-RPC प्रोटोकॉलद्वारे संवाद साधेल.


## सारांश

या धड्यात आपण शिकलात:

1. **A2A प्रोटोकॉल काय आहे** — एक खुला मानक एजंट-टू-एजंट शोध, संदेशवहन,
   आणि कार्य व्यवस्थापनासाठी.
2. **विशेषीकृत एजंट कसे तयार करायचे** — एक चलन विनिमय एजंट, एक क्रियाकलाप नियोजक एजंट,
   आणि एक प्रवास व्यवस्थापक समन्वयक.
3. **वर्कफ्लोमध्ये एजंट कसे जोडायचे** — `WorkflowBuilder` वापरून क्रमिक
   एजंट्सदरम्यान संदेश देवाणघेवाण मॉडेल करण्यासाठी.
4. **उत्पादनात A2A कसे कार्य करते** — डायनॅमिक डिस्कव्हरी आणि स्ट्रीमिंग अपडेट्ससह फ्रेमवर्क-ओलांडणारे, सेवा-ओलांडणारे सहकार्य सक्षम करणे.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला गेला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेची त्रुटी असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत म्हणून विचारला जावा. महत्त्वाची माहिती असल्यास व्यावसायिक मानवी अनुवाद करण्याची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुतींसाठी किंवा चुकीच्या अर्थसोप्यान्साठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
